In [1]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-0.6B")
text = "Hello, building this from scratch"
tokens = tokenizer.tokenize(text)
print(tokens)

config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

['Hello', ',', 'Ġbuilding', 'Ġthis', 'Ġfrom', 'Ġscratch']


In [2]:
import torch
import torch.nn as nn
vocab_size = tokenizer.vocab_size
embedding_dim = 256
embedding = nn.Embedding(vocab_size, embedding_dim)

In [3]:
tokens = tokenizer.encode(
    "Hello, building this from scratch"
)

x = torch.tensor(tokens)

print("Token IDs:", x)
print("Shape:", x.shape)

Token IDs: tensor([ 9707,    11,  4752,   419,   504, 18778])
Shape: torch.Size([6])


In [4]:
x = embedding(x)

print("Embeddings shape:", x.shape)

Embeddings shape: torch.Size([6, 256])


In [5]:
x = torch.tensor(tokens).unsqueeze(0)

print(x.shape)

x = embedding(x)

print(x.shape)

torch.Size([1, 6])
torch.Size([1, 6, 256])


So basically, we have converted to standard (batch, sequence_len, embed_dim) format.

In [6]:
import torch
import torch.nn as nn

B, T, C = x.shape

q_proj = nn.Linear(C, C)
k_proj = nn.Linear(C, C)
v_proj = nn.Linear(C, C)

Q = q_proj(x)
K = k_proj(x)
V = v_proj(x)

print(Q.shape)
print(K.shape)
print(V.shape)

torch.Size([1, 6, 256])
torch.Size([1, 6, 256])
torch.Size([1, 6, 256])


Q = What am I looking for?
K = What information do I contain?
V = What information should I give?

In [7]:
scores = Q @ K.transpose(-2, -1)

print(scores.shape)

torch.Size([1, 6, 6])


In [8]:
attention_weights = torch.softmax(
    scores / (C ** 0.5),
    dim=-1
)

print(attention_weights.shape)
print(attention_weights)

torch.Size([1, 6, 6])
tensor([[[0.1037, 0.0983, 0.1953, 0.2436, 0.2299, 0.1291],
         [0.2143, 0.0870, 0.2019, 0.1182, 0.2127, 0.1658],
         [0.1378, 0.1312, 0.1891, 0.1616, 0.1918, 0.1884],
         [0.1764, 0.2097, 0.1353, 0.1392, 0.1872, 0.1524],
         [0.0877, 0.1394, 0.2941, 0.1921, 0.1283, 0.1583],
         [0.2481, 0.1115, 0.1677, 0.0970, 0.1677, 0.2080]]],
       grad_fn=<SoftmaxBackward0>)


In [9]:
output = attention_weights @ V

print(output.shape)
print(output)

torch.Size([1, 6, 256])
tensor([[[-0.1228, -0.2485, -0.2892,  ...,  0.2785, -0.1094, -0.2355],
         [ 0.0088, -0.2055, -0.2354,  ...,  0.3074, -0.0822, -0.3629],
         [-0.0611, -0.1448, -0.2449,  ...,  0.2545, -0.0252, -0.3449],
         [-0.0742, -0.1149, -0.2613,  ...,  0.2431, -0.0907, -0.4141],
         [-0.0669, -0.1824, -0.2613,  ...,  0.2826,  0.1219, -0.2872],
         [ 0.0674, -0.1116, -0.2076,  ...,  0.2826, -0.0429, -0.4509]]],
       grad_fn=<UnsafeViewBackward0>)


In [10]:
T = x.shape[1]
mask = torch.tril(torch.ones(T, T))
print(mask)

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [11]:
C = 256 # token dimension
heads = 8
head_dim = 32 #splits into 8 heads, each with 32 dimensions

In [12]:
scores = Q @ K.transpose(-2, -1)
scores = scores.masked_fill(
    mask == 0,
    float("-inf")
)
attention_weights = torch.softmax(
    scores / (head_dim ** 0.5),
    dim=-1
)
attention_weights

tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.9277, 0.0723, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2317, 0.2014, 0.5669, 0.0000, 0.0000, 0.0000],
         [0.2766, 0.4512, 0.1306, 0.1416, 0.0000, 0.0000],
         [0.0211, 0.0782, 0.6454, 0.1935, 0.0618, 0.0000],
         [0.4093, 0.0427, 0.1352, 0.0287, 0.1352, 0.2489]]],
       grad_fn=<SoftmaxBackward0>)

In [13]:
out = attention_weights @ V
print(out.shape)

torch.Size([1, 6, 256])


In [14]:
Q = Q.view(B, T, heads, head_dim)
K = K.view(B, T, heads, head_dim)
V = V.view(B, T, heads, head_dim)

In [15]:
Q = Q.transpose(1, 2)
K = K.transpose(1, 2)
V = V.transpose(1, 2)

In [16]:
num_q_heads = 8
num_kv_heads = 2

head_dim = 32

In [17]:
def apply_rope(x, offset=0):
    """
    x shape: [B, heads, T, head_dim]
    """

    B, H, T, D = x.shape

    # Position: offset, offset+1, ..., offset+T-1
    position = torch.arange(
        offset,
        offset + T,
        device=x.device
    )

    # Frequency for each pair of dimensions
    freq = 1.0 / (
        10000 ** (
            torch.arange(
                0,
                D,
                2,
                device=x.device
            ).float() / D
        )
    )

    # [T, D/2]
    angles = position[:, None] * freq[None, :]

    cos = torch.cos(angles)
    sin = torch.sin(angles)

    # Split even and odd dimensions
    x_even = x[..., 0::2]
    x_odd = x[..., 1::2]

    # Rotary transformation
    x_rotated_even = x_even * cos - x_odd * sin
    x_rotated_odd = x_even * sin + x_odd * cos

    # Put dimensions back together
    x_out = torch.zeros_like(x)

    x_out[..., 0::2] = x_rotated_even
    x_out[..., 1::2] = x_rotated_odd

    return x_out

In [18]:
Q = apply_rope(Q)
K = apply_rope(K)

In [19]:
import torch
import torch.nn as nn


class RMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-6):
        super().__init__()

        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):

        # RMS across the last dimension
        rms = torch.sqrt(
            x.pow(2).mean(dim=-1, keepdim=True) + self.eps
        )

        # Normalize
        x = x / rms

        # Learnable scaling
        return self.weight * x

In [20]:
class MLP(nn.Module):

    def __init__(self, dim):
        super().__init__()

        self.fc1 = nn.Linear(dim, 4 * dim)
        self.fc2 = nn.Linear(4 * dim, dim)

    def forward(self, x):
        x = self.fc1(x)
        x = torch.relu(x)
        x = self.fc2(x)
        return x

In [21]:
mlp = nn.Sequential(
    nn.Linear(C, 4 * C),
    nn.GELU(),
    nn.Linear(4 * C, C)
)

out = mlp(x)

print(out.shape)

torch.Size([1, 6, 256])


In [22]:
norm = RMSNorm(C)

x1 = norm(x)

print(x1.shape)

torch.Size([1, 6, 256])


In [23]:
class SelfAttention(nn.Module):

    def __init__(self, C, heads, kv_heads=None):
        super().__init__()

        self.heads = heads
        self.kv_heads = kv_heads or heads
        self.head_dim = C // heads
        self.group_size = self.heads // self.kv_heads

        self.q_proj = nn.Linear(C, C)
        self.k_proj = nn.Linear(C, self.kv_heads * self.head_dim)
        self.v_proj = nn.Linear(C, self.kv_heads * self.head_dim)
        self.out_proj = nn.Linear(C, C)

    def forward(self, x, past_kv=None, use_cache=False):

        B, T, C = x.shape

        offset = past_kv[0].shape[2] if past_kv is not None else 0

        Q = self.q_proj(x)
        K = self.k_proj(x)
        V = self.v_proj(x)

        # Split into heads
        Q = Q.view(B, T, self.heads, self.head_dim).transpose(1, 2)
        K = K.view(B, T, self.kv_heads, self.head_dim).transpose(1, 2)
        V = V.view(B, T, self.kv_heads, self.head_dim).transpose(1, 2)

        # Rotary position embeddings
        Q = apply_rope(Q, offset)
        K = apply_rope(K, offset)

        # Append to KV cache
        if past_kv is not None:
            past_k, past_v = past_kv
            K = torch.cat([past_k, K], dim=2)
            V = torch.cat([past_v, V], dim=2)

        new_kv = (K, V) if use_cache else None

        # Repeat KV heads to match query heads (GQA)
        K_rep = K.repeat_interleave(self.group_size, dim=1)
        V_rep = V.repeat_interleave(self.group_size, dim=1)

        # Attention scores
        scores = Q @ K_rep.transpose(-2, -1)

        # Scale
        scores = scores / (self.head_dim ** 0.5)

        # Causal mask, offset for cached positions
        Tk = K_rep.shape[2]
        mask = torch.tril(
            torch.ones(T, Tk, device=x.device),
            diagonal=Tk - T
        )

        scores = scores.masked_fill(
            mask == 0,
            float("-inf")
        )

        # Softmax
        weights = torch.softmax(scores, dim=-1)

        # Weighted values
        out = weights @ V_rep

        # Combine heads
        out = out.transpose(1, 2).contiguous().view(B, T, C)

        out = self.out_proj(out)

        if use_cache:
            return out, new_kv

        return out

In [24]:
attention = SelfAttention(C=256, heads=8, kv_heads=2)

In [25]:
class TransformerBlock(nn.Module):

    def __init__(self, C, heads, kv_heads=None):
        super().__init__()

        self.norm1 = RMSNorm(C)
        self.attention = SelfAttention(C, heads, kv_heads)

        self.norm2 = RMSNorm(C)
        self.mlp = nn.Sequential(
            nn.Linear(C, 4 * C),
            nn.GELU(),
            nn.Linear(4 * C, C)
        )

    def forward(self, x, past_kv=None, use_cache=False):

        # Attention + residual
        attn_out = self.attention(self.norm1(x), past_kv=past_kv, use_cache=use_cache)

        if use_cache:
            attn_out, new_kv = attn_out

        x = x + attn_out

        # MLP + residual
        x = x + self.mlp(self.norm2(x))

        if use_cache:
            return x, new_kv

        return x

In [26]:
num_layers = 4

blocks = nn.ModuleList([
    TransformerBlock(C=256, heads=8, kv_heads=2)
    for _ in range(num_layers)
])

out = x

for block in blocks:
    out = block(out)

print(out.shape)

torch.Size([1, 6, 256])


In [27]:
lm_head = nn.Linear(C, vocab_size)
logits = lm_head(out)
print(logits.shape)

torch.Size([1, 6, 151643])


In [28]:
next_token_logits = logits[:, -1, :]

In [29]:

class MiniQwen(nn.Module):

    def __init__(
        self,
        vocab_size,
        dim=256,
        num_layers=4,
        num_heads=8,
        num_kv_heads=None
    ):

        super().__init__()

        # Token embedding
        self.embedding = nn.Embedding(
            vocab_size,
            dim
        )

        # Transformer blocks
        self.blocks = nn.ModuleList([
            TransformerBlock(
                dim,
                num_heads,
                num_kv_heads
            )
            for _ in range(num_layers)
        ])

        # Final normalization
        self.norm = RMSNorm(dim)

        # LM Head
        self.lm_head = nn.Linear(
            dim,
            vocab_size,
            bias=False
        )

    def forward(self, tokens, past_kv=None, use_cache=False):

        # tokens:
        # [B, T]

        x = self.embedding(tokens)

        # [B, T, dim]

        new_kvs = [] if use_cache else None

        for i, block in enumerate(self.blocks):

            layer_past = past_kv[i] if past_kv is not None else None

            out = block(x, past_kv=layer_past, use_cache=use_cache)

            if use_cache:
                x, layer_kv = out
                new_kvs.append(layer_kv)
            else:
                x = out

        x = self.norm(x)

        logits = self.lm_head(x)

        # [B, T, vocab_size]

        if use_cache:
            return logits, new_kvs

        return logits

In [30]:
# Model configuration

vocab_size = len(tokenizer)  # includes special tokens (pad/eos), unlike tokenizer.vocab_size

device = "cuda" if torch.cuda.is_available() else "cpu"
num_gpus = torch.cuda.device_count()

model = MiniQwen(
    vocab_size=vocab_size,
    dim=256,
    num_layers=4,
    num_heads=8,
    num_kv_heads=2
)

# Tie the output projection to the input embedding
model.lm_head.weight = model.embedding.weight

model = model.to(device)

if num_gpus > 1:
    model = nn.DataParallel(model)

num_params = sum(p.numel() for p in model.parameters())

print(f"Device: {device} ({num_gpus} GPUs)")
print(f"Parameters: {num_params / 1e6:.1f}M")
print(model)

Device: cuda (2 GPUs)
Parameters: 41.6M
DataParallel(
  (module): MiniQwen(
    (embedding): Embedding(151669, 256)
    (blocks): ModuleList(
      (0-3): 4 x TransformerBlock(
        (norm1): RMSNorm()
        (attention): SelfAttention(
          (q_proj): Linear(in_features=256, out_features=256, bias=True)
          (k_proj): Linear(in_features=256, out_features=64, bias=True)
          (v_proj): Linear(in_features=256, out_features=64, bias=True)
          (out_proj): Linear(in_features=256, out_features=256, bias=True)
        )
        (norm2): RMSNorm()
        (mlp): Sequential(
          (0): Linear(in_features=256, out_features=1024, bias=True)
          (1): GELU(approximate='none')
          (2): Linear(in_features=1024, out_features=256, bias=True)
        )
      )
    )
    (norm): RMSNorm()
    (lm_head): Linear(in_features=256, out_features=151669, bias=False)
  )
)


In [31]:
text = "Hello, building this from scratch"

tokens = tokenizer.encode(text)

x = torch.tensor(
    tokens,
    dtype=torch.long
).unsqueeze(0).to(device)

print("Input shape:", x.shape)

logits = model(x)

print("Logits shape:", logits.shape)

Input shape: torch.Size([1, 6])
Logits shape: torch.Size([1, 6, 151669])


In [32]:
from datasets import load_dataset

target_bytes = 20_000_000  # ~20MB, fits a short training run

dataset = load_dataset(
    "roneneldan/TinyStories",
    split="train",
    streaming=True
)

texts = []
total_bytes = 0

for example in dataset:
    story = example["text"]
    texts.append(story)
    total_bytes += len(story.encode("utf-8"))
    if total_bytes >= target_bytes:
        break

corpus = "\n".join(texts)

print(f"Stories: {len(texts)}")
print(f"Corpus size: {total_bytes / 1e6:.1f} MB")

train_ids = torch.tensor(
    tokenizer.encode(corpus),
    dtype=torch.long
)

print("Training tokens:", train_ids.shape)

README.md: 0.00B [00:00, ?B/s]

Stories: 22473
Corpus size: 20.0 MB


Token indices sequence length is longer than the specified maximum sequence length for this model (4744636 > 131072). Running this sequence through the model will result in indexing errors


Training tokens: torch.Size([4744636])


In [33]:
block_size = 256
batch_size = 16
grad_accum_steps = 2

def get_batch(data, block_size, batch_size, device):

    ix = torch.randint(
        0,
        len(data) - block_size - 1,
        (batch_size,)
    )

    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])

    return x.to(device), y.to(device)

xb, yb = get_batch(train_ids, block_size, batch_size, device)

print(xb.shape, yb.shape)

torch.Size([16, 256]) torch.Size([16, 256])


In [34]:
import time

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))

max_train_seconds = 150  # ~2.5 minutes, leaves budget for compression/SFT/GRPO below
num_steps = 100_000

start_time = time.time()
step = 0

for step in range(num_steps):

    if time.time() - start_time > max_train_seconds:
        break

    optimizer.zero_grad()

    for _ in range(grad_accum_steps):

        xb, yb = get_batch(train_ids, block_size, batch_size, device)

        with torch.amp.autocast("cuda", dtype=torch.float16, enabled=(device == "cuda")):
            logits = model(xb)
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),
                yb.view(-1)
            )

        scaler.scale(loss / grad_accum_steps).backward()

    scaler.step(optimizer)
    scaler.update()

    if step % 20 == 0:
        print(f"step {step}: loss {loss.item():.4f} ({time.time() - start_time:.0f}s)")

print(f"step {step}: loss {loss.item():.4f} ({time.time() - start_time:.0f}s)")

step 0: loss 237.9134 (2s)
step 20: loss 46.2984 (14s)
step 40: loss 38.5767 (26s)
step 60: loss 33.8728 (38s)
step 80: loss 30.1889 (50s)
step 100: loss 28.4919 (62s)
step 120: loss 24.9118 (74s)
step 140: loss 23.0007 (86s)
step 160: loss 20.4797 (98s)
step 180: loss 18.4778 (110s)
step 200: loss 16.4975 (122s)
step 220: loss 15.2155 (134s)
step 240: loss 13.2882 (146s)
step 248: loss 13.4557 (151s)


In [35]:
def generate(model, tokenizer, prompt, max_new_tokens=20):

    model.eval()

    tokens = tokenizer.encode(prompt)
    x = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)

    for _ in range(max_new_tokens):

        logits = model(x)
        next_token_logits = logits[:, -1, :]
        next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)

        x = torch.cat([x, next_token], dim=1)

    model.train()

    return tokenizer.decode(x[0].tolist())

print(generate(model, tokenizer, "Building a language model"))

Building a language model. She. She was a little little little little little little little little little little little little little little


In [36]:
def generate_with_cache(model, tokenizer, prompt, max_new_tokens=20):

    m = model.module if isinstance(model, nn.DataParallel) else model
    m.eval()

    tokens = tokenizer.encode(prompt)
    x = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)

    logits, past_kv = m(x, use_cache=True)
    next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
    generated = [next_token.item()]

    for _ in range(max_new_tokens - 1):

        logits, past_kv = m(next_token, past_kv=past_kv, use_cache=True)
        next_token = torch.argmax(logits[:, -1, :], dim=-1, keepdim=True)
        generated.append(next_token.item())

    m.train()

    return tokenizer.decode(tokens + generated)

print(generate_with_cache(model, tokenizer, "Building a language model"))

Building a language model. She. She was a little little little little little little little little little little little little little little


In [37]:
class Compressor(nn.Module):

    def __init__(self, dim, num_compressed_tokens, heads=8):
        super().__init__()

        self.num_compressed_tokens = num_compressed_tokens
        self.queries = nn.Parameter(torch.randn(num_compressed_tokens, dim) * 0.02)
        self.cross_attn = nn.MultiheadAttention(dim, heads, batch_first=True)
        self.norm = RMSNorm(dim)

    def forward(self, context):

        # context: [B, T, dim] -> compressed: [B, num_compressed_tokens, dim]

        B = context.shape[0]
        queries = self.queries.unsqueeze(0).expand(B, -1, -1)

        compressed, _ = self.cross_attn(queries, context, context)

        return self.norm(compressed)


def run_from_embeds(model, embeds):

    m = model.module if isinstance(model, nn.DataParallel) else model

    x = embeds
    for block in m.blocks:
        x = block(x)
    x = m.norm(x)

    return m.lm_head(x)


context_len = 128
compression_ratio = 8
num_compressed_tokens = context_len // compression_ratio

compressor = Compressor(dim=256, num_compressed_tokens=num_compressed_tokens).to(device)

print(f"Compressor: {context_len} tokens -> {num_compressed_tokens} tokens ({compression_ratio}x)")

Compressor: 128 tokens -> 16 tokens (8x)


In [38]:
# Train the compressor only, base LM stays frozen
for p in model.parameters():
    p.requires_grad = False

compressor_optimizer = torch.optim.AdamW(compressor.parameters(), lr=1e-3)

gen_len = 32
compress_steps = 100

m = model.module if isinstance(model, nn.DataParallel) else model

for step in range(compress_steps):

    starts = torch.randint(0, len(train_ids) - context_len - gen_len - 1, (batch_size,))
    context_ids = torch.stack([train_ids[i:i + context_len] for i in starts]).to(device)
    target_ids = torch.stack([train_ids[i + context_len:i + context_len + gen_len] for i in starts]).to(device)

    with torch.no_grad():
        context_embeds = m.embedding(context_ids)
        target_embeds = m.embedding(target_ids[:, :-1])

    compressed = compressor(context_embeds)
    compressed_input = torch.cat([compressed, target_embeds], dim=1)

    compressed_logits = run_from_embeds(m, compressed_input)
    loss = nn.functional.cross_entropy(
        compressed_logits[:, num_compressed_tokens - 1:, :].reshape(-1, vocab_size),
        target_ids.reshape(-1)
    )

    compressor_optimizer.zero_grad()
    loss.backward()
    compressor_optimizer.step()

    if step % 25 == 0:
        print(f"compressor step {step}: loss {loss.item():.4f}")

for p in model.parameters():
    p.requires_grad = True

compressor step 0: loss 14.0767
compressor step 25: loss 12.2661
compressor step 50: loss 13.7818
compressor step 75: loss 13.2749


In [39]:
# Measure downstream task performance: next-token prediction loss,
# full context vs compressed context
start = torch.randint(0, len(train_ids) - context_len - gen_len - 1, (1,)).item()

context_ids = train_ids[start:start + context_len].unsqueeze(0).to(device)
target_ids = train_ids[start + context_len:start + context_len + gen_len].unsqueeze(0).to(device)

with torch.no_grad():

    full_input = torch.cat([context_ids, target_ids[:, :-1]], dim=1)
    full_logits = m(full_input)
    full_loss = nn.functional.cross_entropy(
        full_logits[:, context_len - 1:, :].reshape(-1, vocab_size),
        target_ids.reshape(-1)
    )

    context_embeds = m.embedding(context_ids)
    compressed = compressor(context_embeds)
    target_embeds = m.embedding(target_ids[:, :-1])
    compressed_input = torch.cat([compressed, target_embeds], dim=1)

    compressed_logits = run_from_embeds(m, compressed_input)
    compressed_loss = nn.functional.cross_entropy(
        compressed_logits[:, num_compressed_tokens - 1:, :].reshape(-1, vocab_size),
        target_ids.reshape(-1)
    )

print(f"Downstream loss predicting the next {gen_len} tokens:")
print(f"  Full context       ({context_len} tok): {full_loss.item():.4f}")
print(f"  Compressed context ({num_compressed_tokens} tok): {compressed_loss.item():.4f}")

Downstream loss predicting the next 32 tokens:
  Full context       (128 tok): 11.5586
  Compressed context (16 tok): 11.7778


In [40]:
sft_examples = [
    ("Continue the story: Once upon a time, there was a", " little rabbit who loved to hop through the meadow."),
    ("Summarize in one word: The cat chased the mouse around the house all day.", " Chase."),
    ("Continue the story: The old wizard opened his book and", " began to read a spell that made the room glow."),
    ("Answer the question: What color is the sky during the day?", " Blue."),
]

def build_sft_batch(examples, tokenizer, device):

    ids_list = []
    label_list = []

    for prompt, response in examples:
        prompt_ids = tokenizer.encode(prompt)
        response_ids = tokenizer.encode(response)

        ids_list.append(prompt_ids + response_ids)
        label_list.append([-100] * len(prompt_ids) + response_ids)

    max_len = max(len(ids) for ids in ids_list)
    pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id or 0

    input_ids = torch.full((len(examples), max_len), pad_id, dtype=torch.long)
    labels = torch.full((len(examples), max_len), -100, dtype=torch.long)

    for i, (ids, labs) in enumerate(zip(ids_list, label_list)):
        input_ids[i, :len(ids)] = torch.tensor(ids)
        labels[i, :len(labs)] = torch.tensor(labs)

    return input_ids.to(device), labels.to(device)

sft_input_ids, sft_labels = build_sft_batch(sft_examples, tokenizer, device)

sft_optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)

for step in range(30):

    logits = model(sft_input_ids[:, :-1])
    loss = nn.functional.cross_entropy(
        logits.reshape(-1, logits.size(-1)),
        sft_labels[:, 1:].reshape(-1),
        ignore_index=-100
    )

    sft_optimizer.zero_grad()
    loss.backward()
    sft_optimizer.step()

    if step % 10 == 0:
        print(f"SFT step {step}: loss {loss.item():.4f}")

print(generate_with_cache(model, tokenizer, "Continue the story: The old wizard opened his book and", max_new_tokens=15))

SFT step 0: loss 14.2508
SFT step 10: loss 9.8086
SFT step 20: loss 6.4954
Continue the story: The old wizard opened his book and the room room room room room room room room room room room room room room


In [41]:
def sample_completion(model, tokenizer, prompt, max_new_tokens, temperature=1.0):

    m = model.module if isinstance(model, nn.DataParallel) else model

    tokens = tokenizer.encode(prompt)
    x = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)

    generated = []
    logprobs = []

    logits, past_kv = m(x, use_cache=True)

    for _ in range(max_new_tokens):

        probs = torch.softmax(logits[:, -1, :] / temperature, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1)
        logprob = torch.log(probs.gather(1, next_token) + 1e-10)

        generated.append(next_token.item())
        logprobs.append(logprob.squeeze())

        logits, past_kv = m(next_token, past_kv=past_kv, use_cache=True)

    return generated, torch.stack(logprobs)


def reward_fn(tokenizer, generated_tokens):

    text = tokenizer.decode(generated_tokens)
    words = text.split()

    return len(set(words)) / max(len(words), 1)


grpo_prompts = [
    "Continue the story: Once upon a time, there was a",
    "Continue the story: The old wizard opened his book and",
]

group_size = 4
grpo_new_tokens = 10
grpo_optimizer = torch.optim.AdamW(model.parameters(), lr=1e-5)

for grpo_step in range(5):

    grpo_optimizer.zero_grad()
    total_loss = 0.0
    mean_reward = 0.0

    for prompt in grpo_prompts:

        rewards = []
        all_logprobs = []

        for _ in range(group_size):
            generated, logprobs = sample_completion(model, tokenizer, prompt, grpo_new_tokens)
            rewards.append(reward_fn(tokenizer, generated))
            all_logprobs.append(logprobs)

        rewards = torch.tensor(rewards, device=device)
        advantages = (rewards - rewards.mean()) / (rewards.std() + 1e-6)

        for logprobs, advantage in zip(all_logprobs, advantages):
            total_loss = total_loss - advantage * logprobs.sum()

        mean_reward += rewards.mean().item() / len(grpo_prompts)

    (total_loss / (len(grpo_prompts) * group_size)).backward()
    grpo_optimizer.step()

    print(f"GRPO step {grpo_step}: mean reward {mean_reward:.3f}, loss {total_loss.item():.4f}")

GRPO step 0: mean reward 0.887, loss 6.2291
GRPO step 1: mean reward 0.663, loss 47.1846
GRPO step 2: mean reward 0.586, loss 28.9158
GRPO step 3: mean reward 0.492, loss 23.3028
GRPO step 4: mean reward 0.690, loss 6.8924
